# 1. tteansformers模型的使用

- 模型使用两种方式
    - 重新训练。
        - 从头训练
        - 从预训练模型训练（微调）
            - torch的原生训练实现
            -  transformers提供训练框架（补充）
    - 使用预训练模型。
        - pipeline：
            - 输入基本上不做预处理
            - 结果需要处理
                - 掌握的推理结果的数据格式。
                    - 目标侦测
                    - 目标分割
                    - （补充）
        - 直接使用模型
            - 预处：掌握处理器
                - 了解模型的推理调用
            - 后处理：掌握处理

## 目标侦测(...)

## 目标分割(...)

## 深度检测(...)

## 图像分类

In [11]:
from transformers import pipeline, AutoModel, AutoConfig, AutoImageProcessor,ViTForImageClassification   # 使用pipeline函数构建ImageClassificationPipeline对象
import torch
# pipe的参数传递个ImageClassificationPipeline构造器

config = AutoConfig.from_pretrained("google/vit-base-patch16-224")   # 字典配置类（可修改）
# config.update({})  #修改配置
processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

pipe = pipeline(
    task="image-classification", 
    model=model,   # "google/vit-base-patch16-224",               # 提交加载   
    config=config, # "google/vit-base-patch16-224s",            # 随着model加载而加载(如果使用模型自带的配置，这个参数是多于) AutoConfig， XXXConfig
    image_processor=processor,   # "google/vit-base-patch16-224",     # 默认，在模型的加载的时候，会自动匹配加载（在pipelie自动加载）
    device="cuda", 
    dtype=torch.bfloat16, # dtype已经包含量化的思想：因为模型训练基本上是torch.float32或者torch.float64。
    trust_remote_code=True, # 模型代码是transformers内置的，该参数设置False。第三方模型的代码随模型发布而发布（开源），在模型总就有py代码，该参数需要设置为True
    use_fast=True,      # 官方提供的模型提供两套预处理机制：（1）Fast：方便（2）：优化：速度快。
    revision="3f49326"
)

pipe("http://images.cocodataset.org/val2017/000000039769.jpg", top_k=10)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cuda


[{'label': 'Egyptian cat', 'score': 0.9374415874481201},
 {'label': 'tabby, tabby cat', 'score': 0.038442619144916534},
 {'label': 'tiger cat', 'score': 0.0144114438444376},
 {'label': 'lynx, catamount', 'score': 0.003274332033470273},
 {'label': 'Siamese cat, Siamese', 'score': 0.0006795947556383908},
 {'label': 'Persian cat', 'score': 0.00011759047629311681},
 {'label': 'snow leopard, ounce, Panthera uncia',
  'score': 0.00010496083996258676},
 {'label': 'cheetah, chetah, Acinonyx jubatus',
  'score': 0.0001018085895339027},
 {'label': 'seat belt, seatbelt', 'score': 9.214074088959023e-05},
 {'label': 'tiger, Panthera tigris', 'score': 6.921369640622288e-05}]

- 代码说明：
    - 能够使用transformers框架调用的模型都是开源：
        - transformers开源
        - 第三方开源

In [6]:
type(pipe.model.vit)

transformers.models.vit.modeling_vit.ViTModel

- pipeline调用的时候，可以设置top_k=3
    - 取结果中，概率最大的三个结果 

- AutoModel加载的时候，要小心其类型：
    - 有的模型有特征模型。加载的时候可能加载到特征模型（不具备分类的能力）。

In [24]:
processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

import PIL.Image as Image
img = Image.open("000000039769.jpg")
# inputs = processor(img)
inputs = processor(img, return_tensor="pt")
print(type(inputs))

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


<class 'transformers.image_processing_base.BatchFeature'>


C:\Program Files\Python313\Lib\site-packages\transformers\image_processing_utils.py:51: UserWarning: The following named arguments are not valid for `ViTImageProcessor.preprocess` and were ignored: 'return_tensor'
  return self.preprocess(images, **kwargs)


In [25]:
inputs

{'pixel_values': [array([[[ 0.11372554,  0.1686275 ,  0.18431377, ..., -0.19215685,
         -0.18431371, -0.18431371],
        [ 0.13725495,  0.1686275 ,  0.18431377, ..., -0.19215685,
         -0.19215685, -0.20784312],
        [ 0.11372554,  0.15294123,  0.16078436, ..., -0.23137254,
         -0.2235294 , -0.21568626],
        ...,
        [ 0.8352941 ,  0.7882353 ,  0.73333335, ...,  0.7019608 ,
          0.64705884,  0.6156863 ],
        [ 0.827451  ,  0.79607844,  0.77254903, ...,  0.58431375,
          0.4666667 ,  0.39607847],
        [ 0.81960785,  0.75686276,  0.75686276, ...,  0.07450986,
         -0.05098039, -0.19215685]],

       [[-0.8039216 , -0.8117647 , -0.8117647 , ..., -0.8901961 ,
         -0.8901961 , -0.8980392 ],
        [-0.7882353 , -0.7882353 , -0.7882353 , ..., -0.8745098 ,
         -0.8745098 , -0.88235295],
        [-0.8117647 , -0.8039216 , -0.7882353 , ..., -0.8901961 ,
         -0.8901961 , -0.8901961 ],
        ...,
        [-0.27058822, -0.31764704, -

In [26]:
import torch
outputs = model(pixel_values=torch.tensor(inputs["pixel_values"][0]).unsqueeze(0))
type(outputs)

transformers.modeling_outputs.ImageClassifierOutput

In [27]:
outputs.logits

tensor([[-2.7440e-01,  8.2152e-01, -8.3646e-02,  4.1588e-01,  5.6233e-01,
          1.8593e-01, -5.7729e-01, -4.6004e-01, -5.3389e-01,  2.4017e-01,
         -3.1957e-01, -5.9910e-01, -6.6402e-01, -4.9756e-01, -6.2448e-01,
         -1.3501e+00, -1.0016e-01, -6.2170e-01,  1.1088e-01, -1.1060e+00,
         -2.0846e-01,  3.1697e-01, -9.3153e-01, -3.0693e-01, -1.0124e+00,
         -1.8751e-01,  5.8825e-01, -3.6161e-01, -7.4696e-01,  7.4135e-01,
         -3.6653e-01, -2.7586e-01,  3.6596e-01, -1.1206e+00, -8.8843e-02,
         -1.1328e+00,  1.5458e-01, -1.0399e+00,  1.0136e+00, -1.0395e+00,
         -2.4214e+00,  5.1125e-01,  4.9458e-01, -7.4005e-01, -1.5815e+00,
         -3.2451e-01, -2.0448e+00, -4.8128e-01, -6.3616e-01, -1.1355e+00,
         -1.0902e+00, -4.5294e-02, -6.4045e-01, -2.3987e-01,  1.3110e-01,
         -1.2664e+00, -4.7160e-01, -4.3717e-01, -9.5664e-01, -5.9685e-01,
          5.0885e-01, -8.4835e-02,  2.6987e-01, -1.4986e-03, -5.3329e-01,
          1.8375e-02,  1.3334e+00,  2.

In [28]:
outputs.logits.shape

torch.Size([1, 1000])

In [30]:
torch.max(torch.sigmoid(outputs.logits), dim=1)

torch.return_types.max(
values=tensor([1.0000], grad_fn=<MaxBackward0>),
indices=tensor([285]))